In [ ]:
!pip install duckdb plotly -q

import duckdb
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

con = duckdb.connect("techpay.db")
print("Ambiente pronto.")

Ambiente pronto.


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

RANDOM_SEED = 123
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

N = 30_000
START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2025, 12, 31)

segments = np.random.choice(['Premium', 'Standard', 'High-Risk'], size=N, p=[0.55, 0.35, 0.10])
credit_score = np.clip(np.random.normal(640, 140, N).astype(int), 300, 900)

channel = np.random.choice(['app', 'web', 'pos', 'atm'], size=N, p=[0.40, 0.30, 0.20, 0.10])
merchant_category = np.random.choice(
    ['varejo', 'viagem', 'eletronico', 'alimentacao', 'servicos', 'saude'],
    size=N, p=[0.30, 0.10, 0.15, 0.20, 0.15, 0.10]
)
transaction_type = np.random.choice(['compra', 'saque', 'transferencia', 'pagamento'], size=N, p=[0.55, 0.15, 0.20, 0.10])

amount = np.clip(np.random.lognormal(4.3, 1.3, N), 5, 40_000)
dates = [START_DATE + timedelta(days=random.randint(0, (END_DATE-START_DATE).days),
                                  hours=random.randint(0, 23)) for _ in range(N)]
risk_score = np.random.uniform(0, 100, N)

is_fraud = []
for i in range(N):
    base = {'Premium': 0.004, 'Standard': 0.012, 'High-Risk': 0.045}[segments[i]]
    if channel[i] == 'app':
        base *= 2.2
    if merchant_category[i] == 'viagem':
        base *= 2.8
    if dates[i].hour < 5:
        base *= 1.8
    if risk_score[i] > 80:
        base *= 1.5
    is_fraud.append(random.random() < min(base, 0.35))

status = ['declined' if f or random.random() < 0.08 else 'approved' for f in is_fraud]

df = pd.DataFrame({
    'transaction_id': range(1, N+1),
    'customer_id': np.random.randint(1, 8000, N),
    'amount': amount.round(2),
    'transaction_type': transaction_type,
    'channel': channel,
    'merchant_category': merchant_category,
    'timestamp': dates,
    'status': status,
    'risk_score': risk_score.round(2),
    'segment': segments,
    'credit_score': credit_score,
    'is_fraud': is_fraud,
})
df = df.sort_values('timestamp').reset_index(drop=True)

df.to_csv('avaliacao_transactions.csv', index=False)

print(f"OK - {len(df)} linhas geradas")
print(f"Taxa de fraude geral: {df['is_fraud'].mean():.2%}")
print(df.groupby('channel')['is_fraud'].mean().sort_values(ascending=False))
print(df.groupby('merchant_category')['is_fraud'].mean().sort_values(ascending=False))

OK - 30000 linhas geradas
Taxa de fraude geral: 2.50%
channel
app    0.037850
atm    0.017958
web    0.017437
pos    0.014243
Name: is_fraud, dtype: float64
merchant_category
viagem         0.053156
saude          0.024737
alimentacao    0.023092
varejo         0.022438
servicos       0.019668
eletronico     0.019595
Name: is_fraud, dtype: float64


In [ ]:
!ls -la avaliacao_transactions.csv

-rw-r--r-- 1 root root 2742011 Sep  8 21:37 avaliacao_transactions.csv


In [ ]:
import duckdb

con = duckdb.connect("techpay.db")
con.execute("""
    CREATE OR REPLACE TABLE raw_transactions AS
    SELECT * FROM read_csv_auto('avaliacao_transactions.csv')
""")

n_linhas = con.execute("SELECT COUNT(*) FROM raw_transactions").fetchone()[0]
print(f"Linhas carregadas no DuckDB: {n_linhas}")
print(con.execute("DESCRIBE raw_transactions").fetchdf())

Linhas carregadas no DuckDB: 30000
          column_name column_type null   key default extra
0      transaction_id      BIGINT  YES  None    None  None
1         customer_id      BIGINT  YES  None    None  None
2              amount      DOUBLE  YES  None    None  None
3    transaction_type     VARCHAR  YES  None    None  None
4             channel     VARCHAR  YES  None    None  None
5   merchant_category     VARCHAR  YES  None    None  None
6           timestamp   TIMESTAMP  YES  None    None  None
7              status     VARCHAR  YES  None    None  None
8          risk_score      DOUBLE  YES  None    None  None
9             segment     VARCHAR  YES  None    None  None
10       credit_score      BIGINT  YES  None    None  None
11           is_fraud     BOOLEAN  YES  None    None  None


In [ ]:
import os
os.makedirs('data/particionado', exist_ok=True)

In [ ]:
con.execute("""
    COPY (
        SELECT *, strftime(timestamp, '%Y-%m') AS ano_mes
        FROM raw_transactions
    )
    TO 'data/particionado/' (FORMAT PARQUET, PARTITION_BY (ano_mes), OVERWRITE_OR_IGNORE 1)
""")
print("Dados particionados salvos em data/particionado/")

Dados particionados salvos em data/particionado/


In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE bronze AS
    SELECT * FROM read_parquet('data/particionado/*/*.parquet')
""")

antes = con.execute("SELECT COUNT(*) FROM bronze").fetchone()[0]

# Remove duplicatas de transaction_id
con.execute("""
    CREATE OR REPLACE TABLE bronze AS
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY transaction_id ORDER BY timestamp) AS rn
        FROM bronze
    ) WHERE rn = 1
""")
con.execute("ALTER TABLE bronze DROP COLUMN rn")

# Remove valores impossíveis
con.execute("""
    CREATE OR REPLACE TABLE bronze AS
    SELECT * FROM bronze
    WHERE amount > 0
      AND risk_score BETWEEN 0 AND 100
      AND credit_score BETWEEN 300 AND 900
""")

depois = con.execute("SELECT COUNT(*) FROM bronze").fetchone()[0]
print(f"Linhas antes da limpeza: {antes} | depois: {depois} | removidas: {antes - depois}")

Linhas antes da limpeza: 30000 | depois: 30000 | removidas: 0


In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE silver AS
    SELECT
        *,
        CASE
            WHEN amount < 100 THEN 'baixo'
            WHEN amount < 1000 THEN 'medio'
            ELSE 'alto'
        END AS faixa_valor,
        CASE
            WHEN EXTRACT(hour FROM timestamp) BETWEEN 5 AND 11 THEN 'manha'
            WHEN EXTRACT(hour FROM timestamp) BETWEEN 12 AND 17 THEN 'tarde'
            WHEN EXTRACT(hour FROM timestamp) BETWEEN 18 AND 23 THEN 'noite'
            ELSE 'madrugada'
        END AS periodo_dia,
        dayname(timestamp) AS dia_semana
    FROM bronze
""")
print(con.execute("SELECT * FROM silver LIMIT 5").fetchdf())

   transaction_id  customer_id  amount transaction_type channel  \
0              12         5113  123.25    transferencia     atm   
1              20         6338   51.09           compra     app   
2              36          689   36.84           compra     app   
3              68         7497  485.45           compra     pos   
4              69         5122   28.02           compra     web   

  merchant_category           timestamp    status  risk_score   segment  \
0            varejo 2025-08-12 02:00:00  approved       29.63  Standard   
1            varejo 2025-05-30 13:00:00  approved        8.35   Premium   
2            viagem 2025-09-19 01:00:00  approved       65.93   Premium   
3            viagem 2025-12-29 02:00:00  approved        1.75   Premium   
4       alimentacao 2025-02-12 13:00:00  approved        8.23   Premium   

   credit_score  is_fraud  ano_mes faixa_valor periodo_dia dia_semana  
0           547     False  2025-08       medio   madrugada    Tuesday  
1 

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE gold_risco_canal_categoria AS
    SELECT
        channel,
        merchant_category,
        COUNT(*) AS total_transacoes,
        ROUND(AVG(risk_score), 2) AS risco_medio,
        ROUND(100.0 * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) / COUNT(*), 3) AS taxa_fraude_pct,
        ROUND(SUM(amount), 2) AS valor_total
    FROM silver
    GROUP BY channel, merchant_category
    ORDER BY taxa_fraude_pct DESC
""")

con.execute("""
    CREATE OR REPLACE TABLE gold_segmento_tempo AS
    SELECT
        segment,
        CASE
            WHEN credit_score < 500 THEN 'baixo'
            WHEN credit_score < 700 THEN 'medio'
            ELSE 'alto'
        END AS faixa_credit_score,
        strftime(timestamp, '%Y-%m-%d') AS dia,
        COUNT(*) AS total_transacoes,
        ROUND(SUM(amount), 2) AS valor_total
    FROM silver
    GROUP BY segment, faixa_credit_score, dia
    ORDER BY dia
""")

print(con.execute("SELECT * FROM gold_risco_canal_categoria LIMIT 10").fetchdf())

  channel merchant_category  total_transacoes  risco_medio  taxa_fraude_pct  \
0     app            viagem              1237        50.73            8.003   
1     app       alimentacao              2409        49.89            3.819   
2     pos            viagem               612        50.37            3.758   
3     atm            viagem               269        53.00            3.717   
4     app             saude              1189        49.85            3.448   
5     app            varejo              3600        49.70            3.250   
6     web            viagem               892        50.62            3.139   
7     app        eletronico              1777        48.96            3.095   
8     app          servicos              1809        50.19            2.819   
9     web             saude               849        49.22            2.473   

   valor_total  
0    195332.68  
1    405844.95  
2    127360.66  
3     53013.46  
4    202261.00  
5    603972.70  
6    170250

In [ ]:
df_canal = con.execute("""
    SELECT channel,
           ROUND(AVG(risk_score), 2) AS risco_medio,
           ROUND(100.0 * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) / COUNT(*), 3) AS taxa_fraude_pct
    FROM silver GROUP BY channel ORDER BY taxa_fraude_pct DESC
""").fetchdf()
print(df_canal)

  channel  risco_medio  taxa_fraude_pct
0     app        49.82            3.785
1     atm        49.50            1.796
2     web        50.36            1.744
3     pos        50.59            1.424


In [ ]:
df_categoria = con.execute("""
    SELECT merchant_category,
           ROUND(AVG(risk_score), 2) AS risco_medio,
           ROUND(100.0 * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) / COUNT(*), 3) AS taxa_fraude_pct
    FROM silver GROUP BY merchant_category ORDER BY taxa_fraude_pct DESC
""").fetchdf()
print(df_categoria)

  merchant_category  risco_medio  taxa_fraude_pct
0            viagem        50.82            5.316
1             saude        50.02            2.474
2       alimentacao        49.69            2.309
3            varejo        50.31            2.244
4          servicos        49.92            1.967
5        eletronico        50.01            1.959


In [ ]:
df_pergunta = con.execute("""
    SELECT periodo_dia,
           COUNT(*) AS total_transacoes,
           ROUND(100.0 * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) / COUNT(*), 3) AS taxa_fraude_pct
    FROM silver
    WHERE channel = 'app' AND merchant_category = 'viagem'
    GROUP BY periodo_dia
    ORDER BY taxa_fraude_pct DESC
""").fetchdf()
print(df_pergunta)

  periodo_dia  total_transacoes  taxa_fraude_pct
0   madrugada               249            9.237
1       manha               370            8.919
2       noite               320            8.125
3       tarde               298            5.705


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

kpi_fraude_geral = con.execute(
    "SELECT ROUND(100.0 * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) / COUNT(*), 3) FROM silver"
).fetchone()[0]

df_temporal = con.execute("""
    SELECT dia_semana,
           COUNT(*) AS total_transacoes,
           ROUND(100.0 * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) / COUNT(*), 3) AS taxa_fraude_pct
    FROM silver GROUP BY dia_semana
""").fetchdf()

df_canal_vol = con.execute("SELECT channel, COUNT(*) AS total FROM silver GROUP BY channel").fetchdf()

fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "indicator"}, {"type": "xy"}],
           [{"type": "domain"}, {"type": "table"}]],
    subplot_titles=("Taxa de fraude geral", "Transações por dia da semana",
                     "Composição por canal", "Top categorias por risco")
)

fig.add_trace(go.Indicator(mode="number", value=kpi_fraude_geral, number={"suffix": "%"}), row=1, col=1)
fig.add_trace(go.Bar(x=df_temporal["dia_semana"], y=df_temporal["total_transacoes"]), row=1, col=2)
fig.add_trace(go.Pie(labels=df_canal_vol["channel"], values=df_canal_vol["total"]), row=2, col=1)
fig.add_trace(go.Table(
    header=dict(values=list(df_categoria.columns)),
    cells=dict(values=[df_categoria[c] for c in df_categoria.columns])
), row=2, col=2)

fig.update_layout(height=800, width=1000, title_text="Dashboard de Risco — TechPay")
fig.write_html("dashboard.html")
print("Dashboard salvo em dashboard.html")
fig.show()

Dashboard salvo em dashboard.html
